##### `Command()` as a Control Primitive

**`Command()` is a control primitive.**

It is a LangGraph control primitive that allows a node to **update state** and **dynamically control where the graph should execute next**.

##### Quick Summary

**Workflow** = overall execution structure  
**Agent** = decision-making component  
**Command** = mechanism to control graph execution

`Command()` allows a node to:

**update the state + choose the next node + sometimes resume an interrupted workflow**

in a single return.

##### Normal Workflow

In a normal workflow, routing is defined in the graph definition:

```text
Node A
  ↓
Conditional Edge
  ↓
Node B / Node C
```

In [ ]:
# Example function
def router(state):
    return state["route"]

builder.add_conditional_edges(
    "router",
    router,
    {
        "billing": "billing",
        "technical": "technical"
    }
)

Here, the node only updates the state, and the conditional edge handles routing separately.

With Command(), the same node can specify:

In [ ]:
return Command(
    goto="billing",
    update={
        "active_agent": "billing"
    }
)

##### `Command()` Control Flow

```text
Current Node
     │
     ↓
 Command()
     │
     ├── update state
     │
     └── goto next node
```

This means **routing and state update happen in the same place**.

##### Difference from a Workflow

**Normal predefined workflow:**

```text
START
  │
  ↓
  A
  │
  ↓
  B
  │
  ↓
  C
  │
  ↓
 END
```

The developer defines the path while building the graph.

**`Command()`:**

```text
Node A
  │
  ↓
Runtime Decision
  │
  ├── Command(goto="B")
  │
  └── Command(goto="C")
```

So `Command()` makes dynamic control flow easier.

##### LLM + `Command()`

Suppose the LLM decides:

```text
User Request
     │
     ↓
    LLM
     │
     ↓
"Billing agent should handle this"
     │
     ↓
Command(goto="billing_agent")
```

Here:

**LLM** = decision maker  
**`Command()`** = mechanism for applying the decision to graph execution

This distinction is important.

> **`Command()` is not an agent.**

The agent makes the decision; `Command()` controls the graph based on that decision.

##### Especially Useful in a Handoff

A **handoff** means transferring control from one agent to another specialized agent so that the new agent continues handling the task.

In [ ]:
return Command(
    goto=decision.target,
    update={
        "active_agent": decision.target
    }
)

##### Triage Agent Handoff Using `Command()`

```text
          Triage Agent
               │
               ↓
          LLM decides:
        technical_agent
               │
               ↓
           Command()
               │
               ├── active_agent = technical_agent
               │
               └── goto technical_agent
               │
               ↓
        Technical Agent
```

##### Roughly the Same Thing with a Conditional Edge

```text
          Triage
             │
             ↓
       Update State
             │
             ↓
     Routing Function
             │
             ↓
    Conditional Edge
             │
             ↓
     Technical Agent
```

**`Command()`:**

```text
          Triage
             │
             ↓
  Command(update + goto)
             │
             ↓
     Technical Agent
```

##### `Command` vs `Send`

**`Command`**  
→ move control to the next node  
→ usually one routing/action  
→ can also update the state  

**`Send`**  
→ dynamically create multiple executions  
→ send custom input to each worker

##### `Command` vs `Send` Example

```python
Command(
    goto="technical_agent"
)
```

means:

```text
Go to technical_agent
```

Whereas:

```python
[
    Send("worker", {"topic": "RAG"}),
    Send("worker", {"topic": "Agents"}),
    Send("worker", {"topic": "Memory"})
]
```

means:

```text
Run worker three times
with three different inputs
```

##### One More Use: HITL Resume

An important use of `Command()`:

```python
Command(
    resume="approved"
)
```

It is also used to **resume an interrupted graph**.

So mentally:

```text
              Command()
                  │
      ┌───────────┼───────────┐
      ↓           ↓           ↓
    goto        update      resume
      │           │           │
where next?   what state   resume interrupted
              changes?       workflow
```